In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"


# Feature Prep Lagged Monthly
Build a reusable feature dataset with lagged, temporal, and monthly yearly-index features, then save it for downstream training notebooks.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
# ---- Paths and config ----
BASE_DATASET_PATH = PROJECT_ROOT / 'training_data_with_features.zarr'
MONTHLY_INDEX_SOURCE_PATH = PROJECT_ROOT / 'full_dataset_20m_monthly_with_indices.zarr'
SPLIT_PATH = PROJECT_ROOT / 'data_split.npz'

OUTPUT_DATASET_PATH = PROJECT_ROOT / 'feature_cache_lagged_monthly.zarr'
OUTPUT_METADATA_PATH = PROJECT_ROOT / 'feature_cache_lagged_monthly_metadata.json'
OUTPUT_VALIDATION_PATH = PROJECT_ROOT / 'feature_cache_lagged_monthly_validation.json'

YEARLY_INDEX_FEATURES = [
    'ndvi_cv_year',
    'ndvi_max_m2m_drop_year',
    'ndvi_max_year',
    'ndvi_min_year',
    'ndvi_std_year',
    'ndwi_cv_year',
    'ndwi_max_m2m_drop_year',
    'ndwi_max_year',
    'ndwi_min_year',
    'ndwi_std_year',
]

for p in [BASE_DATASET_PATH, SPLIT_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Required input not found: {p}')

print(f'Base dataset      : {BASE_DATASET_PATH}')
print(f'Monthly source    : {MONTHLY_INDEX_SOURCE_PATH} (optional)')
print(f'Split file        : {SPLIT_PATH}')
print(f'Output dataset    : {OUTPUT_DATASET_PATH}')
print(f'Output metadata   : {OUTPUT_METADATA_PATH}')
print(f'Output validation : {OUTPUT_VALIDATION_PATH}')

In [ ]:
ds = xr.open_dataset(BASE_DATASET_PATH, engine='zarr')
split_data = np.load(SPLIT_PATH)
train_pixel_indices = split_data['train_pixel_indices']
val_pixel_indices = split_data['val_pixel_indices']
test_pixel_indices = split_data['test_pixel_indices']

print('Loaded base dataset and split file')
print(f'  pixels={ds.sizes.get("pixel", 0):,}, years={ds.sizes.get("year", 0):,}')
print(f'  train={len(train_pixel_indices):,}, val={len(val_pixel_indices):,}, test={len(test_pixel_indices):,}')
print(f'  variables={len(ds.data_vars)}')

In [ ]:
def ensure_delta_feature(ds_in, source_var, delta_var):
    if delta_var in ds_in.data_vars:
        return ds_in
    if source_var not in ds_in.data_vars:
        return ds_in

    arr = ds_in[source_var].astype(np.float32).values
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D variable for {source_var}, got shape {arr.shape}')

    delta = np.zeros_like(arr, dtype=np.float32)
    if arr.shape[1] > 1:
        delta[:, 1:] = arr[:, 1:] - arr[:, :-1]
    delta = np.where(np.isfinite(delta), delta, 0.0).astype(np.float32)
    ds_in[delta_var] = (['pixel', 'year'], delta)
    print(f'Added missing delta feature: {delta_var}')
    return ds_in

def ensure_temporal_features(ds_in):
    required = {'years_since_last_disturbance', 'log_years_since_last_disturbance', 'ever_disturbed'}
    if required.issubset(set(ds_in.data_vars)):
        return ds_in

    if 'disturbances' not in ds_in.data_vars:
        raise ValueError('disturbances variable is required to compute temporal features')

    disturbances = ds_in['disturbances'].values
    if disturbances.ndim != 2:
        raise ValueError(f'Expected disturbances shape (pixel, year), got {disturbances.shape}')

    n_pixels, n_years = disturbances.shape
    years_since = np.full((n_pixels, n_years), fill_value=n_years, dtype=np.float32)
    ever_disturbed = np.zeros((n_pixels, n_years), dtype=np.float32)

    for t in range(n_years):
        if t == 0:
            years_since[:, t] = float(n_years)
            ever_disturbed[:, t] = 0.0
            continue

        past = disturbances[:, :t]
        past_mask = past == 1
        any_past = past_mask.any(axis=1)
        ever_disturbed[:, t] = any_past.astype(np.float32)

        last_disturbance_idx = np.where(
            any_past,
            (past_mask * np.arange(1, t + 1, dtype=np.int32)).max(axis=1) - 1,
            -1,
        )
        years_since[:, t] = np.where(any_past, t - last_disturbance_idx, n_years).astype(np.float32)

    ds_in['years_since_last_disturbance'] = (['pixel', 'year'], years_since)
    ds_in['log_years_since_last_disturbance'] = (['pixel', 'year'], np.log1p(years_since).astype(np.float32))
    ds_in['ever_disturbed'] = (['pixel', 'year'], ever_disturbed)
    print('Added missing temporal features: years_since_last_disturbance, log_years_since_last_disturbance, ever_disturbed')
    return ds_in

def merge_monthly_yearly_indices(base_ds, source_ds, feature_names):
    common_features = [f for f in feature_names if f in source_ds.data_vars]
    if not common_features:
        print('No yearly index features found in source dataset')
        return base_ds, []

    if 'pixel' not in base_ds.coords or 'pixel' not in source_ds.coords:
        raise ValueError('Both datasets must have a pixel coordinate for merging')

    if np.array_equal(base_ds['pixel'].values, source_ds['pixel'].values):
        years = base_ds['year'].values
        for feat in common_features:
            if feat in base_ds.data_vars:
                continue
            aligned = source_ds[feat].sel(year=years).astype(np.float32)
            base_ds[feat] = aligned
        return base_ds, common_features

    key_cols = ['cube_idx', 'x', 'y']
    for col in key_cols:
        if col not in base_ds.data_vars or col not in source_ds.data_vars:
            raise ValueError('Pixel coordinates differ and key columns cube_idx/x/y are unavailable for alignment')

    base_key = pd.MultiIndex.from_arrays([
        base_ds['cube_idx'].values,
        base_ds['x'].values,
        base_ds['y'].values,
    ])
    source_key = pd.MultiIndex.from_arrays([
        source_ds['cube_idx'].values,
        source_ds['x'].values,
        source_ds['y'].values,
    ])

    source_positions = pd.Series(np.arange(len(source_key), dtype=np.int64), index=source_key)
    mapped = source_positions.reindex(base_key)
    matched = mapped.notna().values
    mapped_idx = mapped.fillna(-1).astype(np.int64).values

    years = base_ds['year'].values
    merged = []
    for feat in common_features:
        if feat in base_ds.data_vars:
            continue

        source_arr = source_ds[feat].sel(year=years).astype(np.float32).values
        target = np.full((base_ds.sizes['pixel'], base_ds.sizes['year']), np.nan, dtype=np.float32)
        target[matched] = source_arr[mapped_idx[matched]]
        base_ds[feat] = (['pixel', 'year'], target)
        merged.append(feat)

    print(f'Merged {len(merged)} yearly index features using key alignment')
    return base_ds, merged

In [ ]:
# Ensure lagged/temporal feature components used by training are present
ds_out = ds.copy()

ds_out = ensure_delta_feature(ds_out, 'ndvi', 'ndvi_delta')
ds_out = ensure_delta_feature(ds_out, 'ndwi', 'ndwi_delta')
ds_out = ensure_delta_feature(ds_out, 'nbr', 'nbr_delta')
ds_out = ensure_temporal_features(ds_out)

monthly_merged = []
if MONTHLY_INDEX_SOURCE_PATH.exists():
    monthly_source = xr.open_dataset(MONTHLY_INDEX_SOURCE_PATH, engine='zarr')
    ds_out, monthly_merged = merge_monthly_yearly_indices(ds_out, monthly_source, YEARLY_INDEX_FEATURES)
    monthly_source.close()
else:
    print(f'Monthly index source not found, skipping merge: {MONTHLY_INDEX_SOURCE_PATH}')

print(f'Output variable count: {len(ds_out.data_vars)}')
print(f'Monthly yearly features merged: {len(monthly_merged)}')

In [ ]:
required_vars = [
    's2_bands',
    'dem',
    'disturbances',
    'ndvi',
    'ndwi',
    'ndvi_delta',
    'ndwi_delta',
    'years_since_last_disturbance',
    'log_years_since_last_disturbance',
    'ever_disturbed',
]

missing_required = [v for v in required_vars if v not in ds_out.data_vars]
if missing_required:
    raise ValueError(f'Missing required variables before write: {missing_required}')

if ds_out.sizes.get('year', 0) < 2:
    raise ValueError('Expected at least 2 years for lagged feature training setup')

feature_order_used_by_training = [
    's2_bands',
    'dem',
    'ndvi',
    'ndwi',
    'prev_year_ndvi',
    'prev_year_ndwi',
    'prev_year_B04',
    'prev_year_B03',
    'prev_year_B06',
    'nbr_if_available',
    'ndvi_delta_prev_year',
    'ndwi_delta_prev_year',
    'nbr_delta_prev_year_if_available',
    'years_since_last_disturbance_if_available',
    'log_years_since_last_disturbance_if_available',
    'ever_disturbed_if_available',
    'yearly_monthly_index_features_if_available',
]

validation_summary = {
    'created_at': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'output_dataset': str(OUTPUT_DATASET_PATH),
    'n_pixels': int(ds_out.sizes.get('pixel', 0)),
    'n_years': int(ds_out.sizes.get('year', 0)),
    'years': [int(y) for y in ds_out['year'].values.tolist()],
    'n_variables': int(len(ds_out.data_vars)),
    'monthly_features_present': [f for f in YEARLY_INDEX_FEATURES if f in ds_out.data_vars],
    'nan_counts_key_vars': {},
}

for var_name in required_vars + ['nbr', 'nbr_delta'] + YEARLY_INDEX_FEATURES:
    if var_name in ds_out.data_vars:
        arr = ds_out[var_name].values
        if np.issubdtype(arr.dtype, np.number):
            validation_summary['nan_counts_key_vars'][var_name] = int(np.isnan(arr).sum())

metadata_payload = {
    'notebook_name': 'feature_prep_lagged_monthly',
    'created_at': validation_summary['created_at'],
    'source_base_dataset': str(BASE_DATASET_PATH),
    'source_monthly_dataset': str(MONTHLY_INDEX_SOURCE_PATH),
    'split_file': str(SPLIT_PATH),
    'output_dataset': str(OUTPUT_DATASET_PATH),
    'dimensions': {k: int(v) for k, v in ds_out.sizes.items()},
    'variables': sorted(list(ds_out.data_vars)),
    'monthly_features_expected': YEARLY_INDEX_FEATURES,
    'monthly_features_present': validation_summary['monthly_features_present'],
    'feature_order_used_by_training': feature_order_used_by_training,
    'split_sizes': {
        'train': int(len(train_pixel_indices)),
        'val': int(len(val_pixel_indices)),
        'test': int(len(test_pixel_indices)),
    },
}

print('Validation checks passed')
print(json.dumps({
    'n_pixels': validation_summary['n_pixels'],
    'n_years': validation_summary['n_years'],
    'n_variables': validation_summary['n_variables'],
    'monthly_features_present': len(validation_summary['monthly_features_present']),
}, indent=2))

In [ ]:
print('Writing output dataset...')
ds_out.to_zarr(OUTPUT_DATASET_PATH, mode='w')

with open(OUTPUT_METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata_payload, f, indent=2)

with open(OUTPUT_VALIDATION_PATH, 'w', encoding='utf-8') as f:
    json.dump(validation_summary, f, indent=2)

print('Saved artifacts:')
print(f'  - {OUTPUT_DATASET_PATH}')
print(f'  - {OUTPUT_METADATA_PATH}')
print(f'  - {OUTPUT_VALIDATION_PATH}')

In [ ]:
# Smoke test: reopen saved dataset and print a short summary
ds_check = xr.open_dataset(OUTPUT_DATASET_PATH, engine='zarr')
print('Smoke test summary:')
print(f'  dims: {dict(ds_check.sizes)}')
print(f'  vars: {len(ds_check.data_vars)}')
print(f'  first 10 vars: {sorted(list(ds_check.data_vars))[:10]}')

train_year_idx = 1 if ds_check.sizes.get('year', 0) > 1 else 0
if 'disturbances' in ds_check.data_vars:
    y_sample = ds_check['disturbances'].isel(year=train_year_idx).values
    y_sample = y_sample[np.isin(y_sample, [0, 1])]
    print(f'  sample year index: {train_year_idx}, valid labels: {len(y_sample):,}')

ds_check.close()
print('Smoke test complete')